# 4.2 自己的圖片資料集

使用資料夾結構載入圖片並訓練 CNN。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

## 1. 下載範例資料

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
data_dir = pathlib.Path(tf.keras.utils.get_file('flower_photos', origin=url, untar=True))
# Some environments extract the archive into an extra nested folder.
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'
print(data_dir)

## 2. 用資料夾建立 Dataset

In [ ]:
img_size = (180, 180)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)
class_names

## 3. 效能優化

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 4. 建立模型

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=img_size + (3,)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. 訓練與評估

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Accuracy Curve')
plt.show()

## 6. 套用自己的資料

把 `data_dir` 改成自己的資料夾路徑，並維持每個類別一個子資料夾即可。